# pandas 之十三：数据合并与导出（merge 实战）

---

这一篇用一个**电商订单统计**的真实场景，把两件事讲透：

1. **合并**：`pd.merge()` —— 相当于 SQL 里的 `JOIN`，把散落在多张表里的信息拼成一张大表
2. **导出**：把结果写成 Excel / CSV / TXT，甚至直接写进 MySQL

## 学习目标

1. 掌握 `merge` 的核心参数：`how` / `on` / `left_on` / `right_on` / `suffixes` / `indicator` / `validate`
2. 说清四种连接方式（`inner` / `left` / `right` / `outer`）的差别
3. 会做**多表链式合并**
4. 认清合并的三个"副作用"：**NaN 引入、类型变化、键重复**
5. 会导出 Excel / CSV / TXT，并知道写 MySQL 的关键点


In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.unicode.east_asian_width', True)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)
print('pandas 版本：', pd.__version__)

# 所有导出文件统一放在本目录下的 生成文件/ 里
OUT_DIR = os.path.join(os.getcwd(), '生成文件')
os.makedirs(OUT_DIR, exist_ok=True)
print('导出目录：', OUT_DIR)


---
# 一、`merge()` 到底是什么

它做的就是 SQL 里的 `JOIN`：**按某个（或某几个）键，把两张表拼在一起**。

```python
pd.merge(
    left,            # 左表（主表）
    right,           # 右表（副表）
    how='inner',     # 连接方式：inner / left / right / outer / cross
    on=None,         # 连接键（左右表同名的列）
    left_on=None,    # 左表的键（两边列名不同时用）
    right_on=None,   # 右表的键
    left_index=False,   # 用左表的索引当键
    right_index=False,  # 用右表的索引当键
    suffixes=('_x','_y'),  # 列名撞车时的后缀
    indicator=False,    # 是否加一列标记"这行来自哪边"
    validate=None,      # 校验连接关系（排查数据质量问题的大杀器）
)
```

先用两张迷你表把四种 `how` 一次性看懂：


In [ ]:
left = pd.DataFrame({'id': [1, 2, 3], 'L': ['a', 'b', 'c']})
right = pd.DataFrame({'id': [2, 3, 4], 'R': ['x', 'y', 'z']})

print('左表 left：')
print(left)
print('\n右表 right：')
print(right)


In [ ]:
print('【inner】内连接：只保留两边都有的 id（交集）')
print(pd.merge(left, right, how='inner', on='id'))

print('\n【left】左连接：左表全保留，右表没有的填 NaN')
print(pd.merge(left, right, how='left', on='id'))

print('\n【right】右连接：右表全保留，左表没有的填 NaN')
print(pd.merge(left, right, how='right', on='id'))

print('\n【outer】外连接：两边全保留（并集），谁缺谁填 NaN')
print(pd.merge(left, right, how='outer', on='id'))


一句话记住：

| `how` | 保留哪些行 | 记忆 |
|---|---|---|
| `'inner'` | 只保留键**能对上**的行 | 交集（默认） |
| `'left'` | **左表全留**，右表对不上就 NaN | 做主表统计时最常用 |
| `'right'` | **右表全留** | 用得少，改成 left 反转一下就行 |
| `'outer'` | 两边**全留** | 查"数据对不对得上" |
| `'cross'` | 笛卡尔积（不做键匹配） | 造组合、做全排列 |


---
# 二、真实场景：4 张表拼成一张报表

需求：某电商要统计用户的消费明细，涉及 4 张表 ——

| 表 | 内容 | 键 |
|---|---|---|
| `users` | 用户基础信息（ID、姓名、注册时间、城市） | `user_id` 唯一 |
| `orders` | 订单（订单 ID、用户 ID、下单时间、支付状态） | `order_id` 唯一 |
| `order_details` | 订单明细（明细 ID、订单 ID、商品 ID、数量、单价） | `detail_id` 唯一 |
| `products` | 商品（商品 ID、名称、分类） | `product_id` 唯一 |

目标：合并出「用户姓名 + 城市 + 订单 ID + 下单时间 + 商品名 + 分类 + 数量 + 单价 + 总金额」，
只保留**已支付**且下单时间在 **2023-01-01 之后**的订单。


In [ ]:
# 1) 用户表
users = pd.DataFrame({
    'user_id': [101, 102, 103, 104, 105],
    'user_name': ['张三', '李四', '王五', '赵六', '孙七'],
    'register_time': pd.date_range('2022-01-01', periods=5, freq='ME'),
    'city': ['北京', '上海', '广州', '深圳', '杭州'],
})

# 2) 订单表（注意：106 这个用户【不在】用户表里；2005 未支付，2002 未支付）
orders = pd.DataFrame({
    'order_id': [2001, 2002, 2003, 2004, 2005, 2006],
    'user_id': [101, 101, 102, 103, 105, 106],
    'order_time': pd.date_range('2023-01-15', periods=6, freq='15D'),
    'pay_status': ['已支付', '未支付', '已支付', '已支付', '未支付', '已支付'],
})

# 3) 订单明细表（2007 这个订单不存在；405 这个商品不在商品表里）
order_details = pd.DataFrame({
    'detail_id': [3001, 3002, 3003, 3004, 3005, 3006, 3007],
    'order_id': [2001, 2001, 2002, 2003, 2003, 2004, 2007],
    'product_id': [401, 402, 403, 401, 404, 402, 405],
    'quantity': [2, 1, 3, 1, 2, 5, 1],
    'price': [199, 299, 99, 199, 399, 299, 499],
})

# 4) 商品表（故意【没有】405 号商品）
products = pd.DataFrame({
    'product_id': [401, 402, 403, 404, 406],
    'product_name': ['无线鼠标', '机械键盘', '蓝牙耳机', '显示器', '打印机'],
    'category': ['外设', '外设', '音频', '办公', '办公'],
})

for nm, t in [('users', users), ('orders', orders), ('order_details', order_details), ('products', products)]:
    print(f'{nm:14} {t.shape}  列：{list(t.columns)}')


> **这份数据是被"埋了雷"的**（这正是它的价值）：
>
> | 雷 | 在哪 | 会怎样 |
> |---|---|---|
> | 订单 2006 属于用户 106，但 106 不在用户表 | `orders` | 以用户为主表做左连接时，**2006 会凭空消失** |
> | 订单 2007 不存在于订单表 | `order_details` | 这条明细在合并中会被丢掉 |
> | 商品 405 不在商品表 | `order_details` / `products` | 会得到 NaN 商品名 |
> | 用户 104、105 没有任何订单 | `users` | 左连接后得到 NaN 订单 |

记住这些雷的位置，下面每一步都能看到它们的"爆炸效果"。


In [ ]:
# 第一步：先按业务条件筛出"有效订单"
paid_orders = orders[(orders['order_time'] >= '2023-01-01') & (orders['pay_status'] == '已支付')]

print('订单表共', len(orders), '条，筛出有效订单', len(paid_orders), '条：')
print(paid_orders[['order_id', 'user_id', 'order_time', 'pay_status']])


---
# 三、第一步合并：用户 × 有效订单

用**左连接**：以用户为主表，保证"没下过单的用户"也留在结果里（这样才知道谁没消费）。

`indicator=` 会额外生成一列，标记每行数据的来源：

| 值 | 含义 |
|---|---|
| `both` | 左右表都匹配上了 |
| `left_only` | 只有左表有（右表没匹配上） |
| `right_only` | 只有右表有 |


In [ ]:
df1 = pd.merge(
    left=users,
    right=paid_orders,
    how='left',
    on='user_id',
    indicator='user_order_merge',
    validate='many_to_many',      # 先记住这里，下面专门讲它为什么不能写 one_to_many
)

print('df1 形状：', df1.shape)
print(df1)


In [ ]:
print('按来源统计：')
print(df1['user_order_merge'].value_counts())


**⚠️ 发现第一颗雷：订单 2006 不见了！**

`paid_orders` 里明明有 2006，可 `df1` 里根本找不到它 ——
因为 2006 属于用户 **106**，而 106 **不在用户表**里。
左连接以 `users` 为准，对不上的右表行就被**静默丢弃**了。

验证一下：


In [ ]:
print('paid_orders 里的订单：', sorted(paid_orders['order_id'].tolist()))
print('df1 里的订单：      ', sorted(df1['order_id'].dropna().astype(int).unique().tolist()))
print('2006 在 paid_orders 里吗？', 2006 in paid_orders['order_id'].values)
print('106 在 users 里吗？      ', 106 in users['user_id'].values)
print('\n结论：2006 是"孤儿订单"（用户不存在），左连接时被丢掉了。')
print('想保留它，就得换成 how="outer"，或者先解决用户表缺 106 的问题。')


In [ ]:
# 顺便看看 outer 连接会怎样（不赋值，只看一眼）
outer_check = pd.merge(users, paid_orders, how='outer', on='user_id')
print('用 outer 连接，2006 就回来了（但城市、姓名是 NaN）：')
print(outer_check[outer_check['order_id'].notna()][['user_id', 'user_name', 'order_id', 'city']])


## 顺带讲透：`validate` 为什么不能写 `'one_to_many'`

`validate` 是用来**校验"连接关系"是否符合预期**的，参数有四种：

| 值 | 要求 | 违规时 |
|---|---|---|
| `'one_to_one'` | 两边键都唯一 | 报 `MergeError` |
| `'one_to_many'` | **左表**键唯一 | 报 `MergeError` |
| `'many_to_one'` | **右表**键唯一 | 报 `MergeError` |
| `'many_to_many'` | 不校验（默认） | 永不报错 |

按业务逻辑，用户对订单是"一对多"（一个用户多个订单），
看起来应该写 `validate='one_to_many'`。但实际会**报错**：


In [ ]:
try:
    pd.merge(users, paid_orders, how='left', on='user_id', validate='one_to_many')
    print('没报错？')
except Exception as e:
    print('报错了：')
    print(f'  {type(e).__name__}: {str(e)[:200]}')


> **这是一个非常值得记住的坑**：
>
> `validate` 判断"左表键是否唯一"时，**会把左连接产生的两个 `NaN` 当成重复键**！
> 用户 104 和 105 都没订单，它们的 `order_id` 都是 `NaN`，
> 两个 `NaN` 在 pandas 眼里"重复了"，于是判定左表键不唯一 → 报错。
>
> 这其实揭示了 `validate` 的一个设计前提：**它假设用来当键的列没有缺失值**。
> 所以在有左连接的链条里，要么用 `many_to_many`，
> 要么**先过滤掉 NaN 行再校验**，要么把键列必填化。
>
> 原稿在这里直接写了 `validate='many_to_many'` —— 结果是对的，
> 但它其实是**被迫**的，不是"随便写写"。这一点原稿没有解释。


In [ ]:
# 想用 one_to_many 校验？先把没有订单的行去掉（键就不含 NaN 了）
no_nan = df1[df1['order_id'].notna()]
print('去掉 NaN 键后，左表键唯一吗？', no_nan['order_id'].is_unique)
print('这时再校验 one_to_many 就通过了：')

checked = pd.merge(
    no_nan,
    order_details,
    how='left',
    on='order_id',
    validate='one_to_many',
)
print('校验通过，形状：', checked.shape)


---
# 四、第二步合并：接上订单明细

一个订单可能对应多条明细（一单多商品），所以行号会**变多**。


In [ ]:
df2 = pd.merge(
    left=df1,
    right=order_details,
    how='left',
    on='order_id',
    indicator='order_detail_merge',
    validate='many_to_many',      # 同样，因为 df1 里还有 NaN 键
)

print('df1 有', len(df1), '行 -> df2 有', len(df2), '行')
print(df2[['user_name', 'order_id', 'detail_id', 'product_id', 'quantity', 'price', 'order_detail_merge']])


---
# 五、第三步合并：接上商品表

明细对商品是"多对一"（多条明细可能买同一个商品），所以这里可以放心写
`validate='many_to_one'` —— 右表（商品表）的 `product_id` 是唯一的。


In [ ]:
final_df = pd.merge(
    left=df2,
    right=products,
    how='left',
    on='product_id',
    indicator='detail_product_merge',
    validate='many_to_one',
)

# 算总金额
final_df['total_amount'] = final_df['quantity'] * final_df['price']

print('最终形状：', final_df.shape)
print(final_df[['user_name', 'order_id', 'product_id', 'product_name', 'category',
                'quantity', 'price', 'total_amount', 'detail_product_merge']])


In [ ]:
print('商品匹配情况：')
print(final_df['detail_product_merge'].value_counts())


> **第二颗雷**：`order_details` 里的 2007 号订单（买了 405 号商品）**也消失了**。
> 原因和 2006 一样 —— 它以 `df1`（只有 2001/2003/2004）为左表，2007 对不上。
>
> 所以原稿结尾那句"无匹配商品的明细数"统计出来的 2 条，
> **其实并不是"商品没匹配上"，而是"这些用户压根没订单"**（赵六、孙七两行）。
> 原稿的数据设计让"商品缺失"这个场景**从未真正出现过**。
>
> 我们下面专门补一个实验，让 405 号商品真的走到最后，看看会发生什么。


In [ ]:
# 补实验：改用 outer 连接，把所有"孤儿数据"都保下来，看看会发生什么
all_orders = orders[orders['order_time'] >= '2023-01-01']     # 只按时间过滤，不管支付状态

t1 = pd.merge(users, all_orders, how='outer', on='user_id')    # 2006 这种孤儿订单保住了
t2 = pd.merge(t1, order_details, how='outer', on='order_id')   # 2007 这种孤儿明细保住了
t3 = pd.merge(t2, products, how='left', on='product_id')       # 商品表用 left 就够

print('outer 合并后共', len(t3), '行\n')

print('① 405 号商品终于走到最后了（它的产品名是 NaN）：')
print(t3[t3['product_id'] == 405][['user_name', 'order_id', 'detail_id', 'product_id',
                                   'product_name', 'category', 'quantity', 'price']])

print('\n② 所有"商品名缺失"的行：')
print(t3[t3['product_name'].isna()][['user_name', 'order_id', 'detail_id', 'product_id']])

print('\n③ 孤儿订单 2006 也回来了（用户信息是 NaN）：')
print(t3[t3['order_id'] == 2006][['user_id', 'user_name', 'order_id', 'product_id']])


> **对比一下就看明白了**：
>
> | 连接方式 | 结果行数 | 2006（孤儿订单） | 405（幽灵商品） |
> |---|---|---|---|
> | `left` 链（前面用的） | 7 行 | ❌ 被丢 | ❌ 被丢 |
> | `outer` 链（这里） | 10 行 | ✅ 保留 | ✅ 保留 |
>
> 这就是 `how` 参数的实际威力：**它决定了"对不上的数据"是丢掉还是保留。**
> 做数据质量排查时（"到底有多少数据对不上？"），`outer` 是你的朋友；
> 出正式报表时，通常用 `left` 保证"主表一条不漏"。


---
# 六、整理成最终报表

三步走：**选列 → 排序 → 填补缺失值**。


In [ ]:
keep_cols = ['user_name', 'city', 'order_id', 'order_time',
             'product_name', 'category', 'quantity', 'price', 'total_amount',
             'user_order_merge', 'detail_product_merge']

result = final_df[keep_cols].sort_values(['user_name', 'order_id'])
print('整理后的结果：')
print(result)


## 注意：合并会让整数列变成小数！

打印上面那张表，你会发现 `order_id` 显示成 `2001.0`。
这不是显示问题，是**真实的数据类型变了** ——
原因：左连接引入了 `NaN`，而 `NaN` 是浮点数，`int` 列为了容纳它，
**整列会被"升格"成 `float`**。


In [ ]:
print('原始 orders 的 order_id 类型：', orders['order_id'].dtype)
print('合并后 result 的 order_id 类型：', result['order_id'].dtype)
print()
print(final_df.dtypes[['order_id', 'quantity', 'price', 'total_amount']])


In [ ]:
# 【解法】换成 pandas 的可空整数类型 Int64（大写 I），既能存 NaN，又保持整数语义
fixed = final_df.copy()
fixed['order_id'] = fixed['order_id'].astype('Int64')
fixed['quantity'] = fixed['quantity'].astype('Int64')
print('换成 Int64 之后：')
print(fixed[['user_name', 'order_id', 'quantity', 'price']].head(4))
print()
print('类型：', fixed['order_id'].dtype, '/', fixed['quantity'].dtype)


## 填补缺失值：`fillna` 用字典给不同列填不同的值


In [ ]:
result_filled = result.copy()
result_filled = result_filled.fillna({
    'order_id': '无订单',
    'order_time': '无订单',
    'product_name': '无匹配商品',
    'category': '无匹配商品',
    'quantity': 0,
    'price': 0,
    'total_amount': 0,
})

print('填完之后打印更友好（不再是一堆 NaN）：')
print(result_filled.to_string(index=False))


In [ ]:
print('=== 关键统计 ===')
print('总用户数：        ', len(users))
print('有效订单数（已支付 + 2023 年后）：', len(paid_orders))
print('有订单的用户数：  ', result_filled[result_filled['order_id'] != '无订单']['user_name'].nunique())
print('没有任何订单的用户：', result_filled[result_filled['order_id'] == '无订单']['user_name'].tolist())
print('订单总金额：      ', result_filled['total_amount'].sum())


---
# 七、其他常用参数速查

### 7.1 列名撞车 → `suffixes`

两张表都有同名列（但**不是**连接键）时，pandas 会自动加后缀 `_x` / `_y`，
想改就得用 `suffixes`。


In [ ]:
a = pd.DataFrame({'id': [1, 2], 'create_time': ['A1', 'A2']})
b = pd.DataFrame({'id': [1, 2], 'create_time': ['B1', 'B2']})

print('默认后缀 _x / _y：')
print(pd.merge(a, b, on='id'))

print('\n自定义后缀：')
print(pd.merge(a, b, on='id', suffixes=('_订单', '_用户')))


### 7.2 两边键名不同 → `left_on` / `right_on`


In [ ]:
u = pd.DataFrame({'uid': [1, 2], 'name': ['甲', '乙']})
o = pd.DataFrame({'user_id': [1, 2], 'amount': [10, 20]})

print('键名不同（uid vs user_id）：')
print(pd.merge(u, o, left_on='uid', right_on='user_id'))

print('\n注意：两个键列都会被保留，通常要再 drop 掉多余的那个：')
print(pd.merge(u, o, left_on='uid', right_on='user_id').drop(columns=['user_id']))


### 7.3 用索引当键 → `left_index` / `right_index`


In [ ]:
u2 = pd.DataFrame({'name': ['甲', '乙']}, index=pd.Index([1, 2], name='uid'))
print('u2 的键在索引上：')
print(u2)

print('\n用 left_index + right_on 连接：')
print(pd.merge(u2, o, left_index=True, right_on='user_id'))


### 7.4 多列当键 → `on=['列1','列2']`


In [ ]:
s1 = pd.DataFrame({'店': ['A', 'A', 'B'], '日期': ['1月', '2月', '1月'], '销量': [1, 2, 3]})
s2 = pd.DataFrame({'店': ['A', 'A', 'B'], '日期': ['1月', '2月', '1月'], '利润': [10, 20, 30]})
print('按【店 + 日期】两个键合并：')
print(pd.merge(s1, s2, on=['店', '日期']))


---
# 八、导出数据

算完的结果总得交出去。常见四种出口：**Excel / CSV / TXT / 数据库**。


## 8.1 导出 Excel

| 引擎 | 特点 |
|---|---|
| `openpyxl`（默认） | 支持 `.xlsx`，够用 |
| `xlsxwriter` | **格式设置更强**（列宽、颜色、数字格式） |
| `xlwt` | 很老的 `.xls` 格式，基本不用了 |

**重要参数**：`index=False`（不要把行号当一列写进去）、`na_rep`（缺失值显示成什么）。


In [ ]:
xlsx_path = os.path.join(OUT_DIR, '电商订单统计结果.xlsx')

result.to_excel(
    excel_writer=xlsx_path,
    sheet_name='订单明细',
    index=False,          # 【关键】不导出索引列，否则会多出一列 0,1,2...
    na_rep='--',          # 缺失值写成 "--"，比 NaN 好看
    engine='openpyxl',
)
print('已导出：', xlsx_path)
print('文件大小：', os.path.getsize(xlsx_path), '字节')


### 一张工作簿放多个 sheet：`ExcelWriter`

想在一个 `.xlsx` 里放"明细 + 汇总"两张表，就用 `ExcelWriter`。
推荐写成 `with` 语句，自动收尾（原稿是手工 `writer.close()`，容易忘）。


In [ ]:
adv_path = os.path.join(OUT_DIR, '电商订单统计_高级版.xlsx')

# 先算一份用户汇总
# 【注意】订单数要先把"无订单"剔掉再 nunique，否则没下单的人也会被数成 1
user_summary = (
    result_filled
    .groupby('user_name', as_index=False)
    .agg(订单数=('order_id', lambda s: s[s != '无订单'].nunique()),
         消费总额=('total_amount', 'sum'))
    .sort_values('消费总额', ascending=False)
    .reset_index(drop=True)
)
print('用户消费汇总：')
print(user_summary)


In [ ]:
# 验证：没有订单的用户，订单数应该是 0
print('检查一下孙七、赵六的订单数：')
print(user_summary[user_summary['user_name'].isin(['孙七', '赵六'])])


> **这里藏着一个很容易犯的错**，值得单独说：
>
> 我一开始写的是 `lambda s: (s != '无订单').nunique()` —— 看着挺合理，
> 结果**没有订单的孙七、赵六也被算成了 1 单**。
>
> 原因：`s != '无订单'` 得到一个**布尔序列**，`nunique()` 数的是
> "这个布尔序列里有几种不同的值"。对孙七来说它是 `[False]`，
> 只有一种值，所以 `nunique()` 返回 **1** —— 而不是"有 1 个 True"。
>
> | 写法 | 孙七的结果 | 对吗 |
> |---|---|---|
> | `(s != '无订单').nunique()` | 1 | ❌ 数的是"有几种布尔值" |
> | `(s != '无订单').sum()` | 0 | ✅ 数的是"有几个 True" |
> | `s[s != '无订单'].nunique()` | 0 | ✅ 先过滤再数"有几个不同订单号" |
>
> 最后一种最严谨（如果同一订单出现两次，`sum()` 会数成 2，`nunique()` 仍然是 1）。
>
> **一句话记住**：想"数个数"用 `sum()`；想"数有几种"才用 `nunique()`。


In [ ]:
with pd.ExcelWriter(adv_path, engine='xlsxwriter') as writer:
    result_filled.to_excel(writer, sheet_name='订单明细', index=False)
    user_summary.to_excel(writer, sheet_name='用户消费汇总', index=False)

    # xlsxwriter 独有的"美化"能力：设置列宽和数字格式
    workbook = writer.book
    money_fmt = workbook.add_format({'num_format': '#,##0.00', 'align': 'right'})
    ws1 = writer.sheets['订单明细']
    ws1.set_column('I:I', 14, money_fmt)     # I 列是 total_amount
    ws2 = writer.sheets['用户消费汇总']
    ws2.set_column('C:C', 14, money_fmt)

print('已导出（含 2 个 sheet + 格式）：', adv_path)


In [ ]:
# 验证一下真的有两个 sheet
print('读回来的 sheet 名：', pd.ExcelFile(adv_path).sheet_names)
print()
print('读回"用户消费汇总"：')
print(pd.read_excel(adv_path, sheet_name='用户消费汇总'))


## 8.2 导出 CSV

CSV 是最通用的交换格式，但有三个坑：

| 坑 | 现象 | 解法 |
|---|---|---|
| 中文乱码 | Excel 打开是乱码 | `encoding='utf-8-sig'` |
| 数据里本来就有逗号 | 列被切错 | 换分隔符 `sep='\t'` 或 `sep='|'` |
| 缺失值 | 显示 `NaN` | `na_rep=''` 或 `na_rep='无'` |


In [ ]:
csv_path = os.path.join(OUT_DIR, '电商订单统计结果.csv')

result_filled.to_csv(
    csv_path,
    index=False,
    na_rep='',
    encoding='utf-8-sig',   # 【Windows 必备】带 BOM，Excel 打开不乱码
    sep=',',
)
print('已导出：', csv_path)

# 看一眼文件开头（注意开头的 BOM 字符 ）
with open(csv_path, encoding='utf-8-sig') as f:
    for _ in range(3):
        print('  ', f.readline().rstrip())


In [ ]:
# 换分隔符：数据里含逗号时，用制表符更安全
tsv_path = os.path.join(OUT_DIR, '电商订单统计_制表符分隔.csv')
result_filled.to_csv(tsv_path, index=False, encoding='utf-8-sig', sep='\t')
print('已导出（制表符分隔）：', tsv_path)

print('\n用 sep="\\t" 读回来：')
print(pd.read_csv(tsv_path, sep='\t').head(3))


## 8.3 导出 TXT

TXT 没有标准格式，两种常见做法：

- **`to_string()`**：固定宽度对齐，给人看漂亮，但**机器不好解析**
- **`to_csv(sep='|')`**：其实存成了 TXT，但结构规整，**机器好解析**


In [ ]:
txt_pretty = os.path.join(OUT_DIR, '电商订单统计结果.txt')
result_filled.to_string(buf=txt_pretty, index=False, na_rep='无', col_space=12)
print('已导出（定宽对齐，给人看）：', txt_pretty)
print()
with open(txt_pretty, encoding='utf-8') as f:
    for _ in range(5):
        print('  ', f.readline().rstrip())


In [ ]:
# 结构化 TXT：用竖线分隔，机器友好
txt_pipe = os.path.join(OUT_DIR, '电商订单统计_竖线分隔.txt')
result_filled.to_csv(txt_pipe, index=False, encoding='utf-8-sig', sep='|', na_rep='无')
print('已导出（竖线分隔，给机器）：', txt_pipe)
print()
print(pd.read_csv(txt_pipe, sep='|').head(3))


## 8.4 写进 MySQL

pandas 用 `to_sql()` 写数据库，需要两样东西：

1. **连接引擎**：`sqlalchemy.create_engine(...)`
2. **告诉 MySQL 用什么类型**：`dtype={列名: SQLAlchemy类型}`

```python
engine = create_engine("mysql+pymysql://用户:密码@主机:端口/库名?charset=utf8mb4")
```

> ⚠️ **安全提醒**：密码写在代码里是**不安全**的（代码一进 git 就泄露了）。
> 正式项目应该用**环境变量**读取：
> `os.getenv('DB_PASSWORD')`。

> ⚠️ **数据安全**：`if_exists='replace'` 会**先删表再建表**！
> 如果那张表里有别的重要数据，就一起没了。
> 下面的示例**默认不写入**，只做连接和类型映射的演示，你确认没问题了再把开关打开。


In [ ]:
from sqlalchemy import create_engine

# 【安全设计】默认关闭写库，避免"跑一次教学笔记就把已有表替换掉"
WRITE_TO_MYSQL = False

DB_CONFIG = {
    'host': 'localhost',
    'port': 3306,
    'user': 'root',
    'password': '123456',          # 练习环境可以，正式环境请用环境变量
    'db': 'window_function',
}

engine = create_engine(
    f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@"
    f"{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['db']}?charset=utf8mb4",
    connect_args={'connect_timeout': 3},
)

try:
    with engine.connect() as conn:
        tables = [r[0] for r in conn.exec_driver_sql('SHOW TABLES')]
    print('连接成功！', DB_CONFIG['db'], '库里现有的表：')
    print(' ', tables)
except Exception as e:
    print('连不上 MySQL（不影响继续学习）:', type(e).__name__, str(e)[:120])
    print('需要本机启动 MySQL 并且把上面的 DB_CONFIG 改成你自己的。')
    engine = None


### 数据预处理（MySQL 对类型比较挑剔）


In [ ]:
df_mysql = result_filled.copy()

# 1) 日期列：把"无订单"这种占位文本变成真正的空值，再转成日期
#    format='mixed' 表示"这一列里格式不统一，请逐个识别"，避免 pandas 报警告
df_mysql['order_time'] = pd.to_datetime(df_mysql['order_time'], format='mixed', errors='coerce')

# 2) 字符串列：限制长度，避免超出字段长度报错
df_mysql['product_name'] = df_mysql['product_name'].astype(str).str[:50]
df_mysql['category'] = df_mysql['category'].astype(str).str[:20]

print('预处理后的类型：')
print(df_mysql.dtypes[['order_id', 'order_time', 'product_name', 'quantity', 'total_amount']])


### `to_sql` 的写入与类型映射

为什么必须手动指定 `dtype`？因为 pandas 的 `int64` / `object` 这些类型
MySQL 不认识，得翻译成 SQL 类型：


In [ ]:
if engine is not None and WRITE_TO_MYSQL:
    from sqlalchemy.types import String, Date, Integer, DECIMAL

    df_mysql.to_sql(
        name='pandas13_order_demo',   # 注意：写到一个【专门的教学表】，不动你原来的表
        con=engine,
        if_exists='replace',          # 表已存在则替换（⚠️ 会先删表！）
        index=False,
        chunksize=1000,               # 分批写入，大数据量时避免一次性占用过多内存
        dtype={
            'user_name': String(20),
            'city': String(20),
            'order_id': String(20),   # 里面含"无订单"文本，所以用字符串
            'order_time': Date(),
            'product_name': String(50),
            'category': String(20),
            'quantity': Integer(),
            'price': DECIMAL(10, 2),
            'total_amount': DECIMAL(10, 2),
        },
    )
    print('写入完成，读回来验证前 3 行：')
    with engine.connect() as conn:
        print(pd.read_sql('SELECT * FROM pandas13_order_demo LIMIT 3', conn))
else:
    print('WRITE_TO_MYSQL =', WRITE_TO_MYSQL, '-> 本次不写库（安全模式）')
    print('\n要真正练习写库，把上面的 WRITE_TO_MYSQL 改成 True 再运行这个单元格。')
    print('写入前请确认：')
    print('  1) 本机 MySQL 已启动，DB_CONFIG 里的账号密码库名正确')
    print('  2) 目标表名不会覆盖你的重要数据（这里用的是 pandas13_order_demo）')


### 导出到 Excel 的多 sheet（等价写法回顾）

原稿最后用 `ExcelWriter` 把同一个 DataFrame 写进两个 sheet：

```python
writer = pd.ExcelWriter('order_statistics.xlsx')
df.to_excel(writer, index=False, sheet_name='order_statistics')
df.to_excel(writer, index=False, sheet_name='order_statistics2')
writer.close()
```

逻辑没错，但**推荐用 `with` 语句**，即使中间报错也能保证文件被正确关闭：


In [ ]:
multi_path = os.path.join(OUT_DIR, 'order_statistics.xlsx')
with pd.ExcelWriter(multi_path) as writer:
    df_mysql.to_excel(writer, index=False, sheet_name='order_statistics')
    df_mysql.to_excel(writer, index=False, sheet_name='order_statistics2')

print('已导出：', multi_path)
print('sheet 名：', pd.ExcelFile(multi_path).sheet_names)


In [ ]:
print('=== 本次导出清单 ===')
for f in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f)
    print(f'  {f:<34} {os.path.getsize(p):>8} 字节')


---
# 常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 左连接"丢数据" | 右表里有、左表没有的键凭空消失 | 确认主表选对；或改 `how='outer'` |
| `validate='one_to_many'` 报错 | 左连接后键列含 NaN，被判重复 | 用 `many_to_many`，或先去掉 NaN 行 |
| 合并后整数变小数 | `2001` 变成 `2001.0` | `astype('Int64')`（大写 I） |
| 用了 `many_to_many` 还不自知 | 结果行数暴涨（笛卡尔积） | 合并后**必看 `shape`** |
| 列名变成 `_x` / `_y` | 不知道哪个是哪个 | 用 `suffixes` 起有意义的后缀 |
| `left_on`/`right_on` 后多出一个键列 | 有两列重复的 ID | 用完 `drop(columns=[...])` |
| `to_excel` 忘了 `index=False` | 多出一列行号 | 养成显式写 `index=False` 的习惯 |
| CSV 在 Excel 打开乱码 | 中文变问号 | `encoding='utf-8-sig'` |
| `to_sql(if_exists='replace')` | **先删表再加**，历史数据全没 | 换成 `'append'`，或写到专用表 |
| 密码硬编码在代码里 | 提交到 git 就泄露 | 用 `os.getenv()` 读环境变量 |
| `ExcelWriter` 忘记 `close()` | 文件损坏 / 打不开 | 用 `with` 语句 |


---
# 动手练习

1. **连接方式实验**：用第一节的 `left` / `right` 两表，分别算出四种 `how` 结果的行数，
   并解释为什么行数不同。
2. **一对一校验**：把 `products` 和它自己按 `product_id` 合并，用
   `validate='one_to_one'`，验证不报错。再故意把 key 改成 `category`
   （一个分类对应多个商品），看 `validate='one_to_one'` 会报什么错。
3. **保住孤儿数据**：改写流程，让 2006 号订单（用户 106）最终出现在报表里。
   提示：可以先用 `how='outer'` 或改用 `orders` 当主表。
4. **用户消费排行**：从最终结果里算出每个用户的订单总金额，按金额降序排列，
   并导出到一个 Excel 的"用户排行" sheet。
5. **编码实验**：分别用 `encoding='utf-8'` 和 `'utf-8-sig'` 导出同一个 CSV，
   用十六进制打开看前几个字节，找出 BOM 到底长什么样。
6. **类型修复**：把 `result_filled['order_id']` 这种"字符串和数字混在一起"的列，
   改成既保留语义、又能安全存 NaN 的写法，并说明为什么 `astype(int)` 会失败。


---
# 答案思路

**1. 四种连接的行数**

```python
for how in ['inner', 'left', 'right', 'outer']:
    r = pd.merge(left, right, how=how, on='id')
    print(how, len(r))
# inner: 2（id=2,3 都在）
# left : 3（1,2,3；id=1 右边没有）
# right: 3（2,3,4；id=4 左边没有）
# outer: 4（并集 1,2,3,4，各自缺的地方是 NaN）
```

**2. 一对一校验**

```python
pd.merge(products, products, on='product_id', validate='one_to_one')   # OK

try:
    pd.merge(products, products, on='category', validate='one_to_one')
except Exception as e:
    print(type(e).__name__, e)
# MergeError: Merge keys are not unique in either left or right dataset
# 因为"外设"分类下有 2 个商品，键不唯一
```

**3. 保住孤儿数据**

```python
# 以 orders 为主表（而不是 users），106 就不会被丢掉
base = pd.merge(all_orders, users, how='left', on='user_id')
print(base[base['order_id'] == 2006])   # user_name 是 NaN，但订单在
```

**4. 用户消费排行**

```python
rank = (result_filled
        .groupby('user_name', as_index=False)['total_amount'].sum()
        .sort_values('total_amount', ascending=False))
with pd.ExcelWriter(os.path.join(OUT_DIR, '用户排行.xlsx')) as w:
    rank.to_excel(w, sheet_name='用户排行', index=False)
print(rank)
```

**5. 编码实验**

```python
p1 = os.path.join(OUT_DIR, 'no_bom.csv')
p2 = os.path.join(OUT_DIR, 'with_bom.csv')
result_filled.to_csv(p1, index=False, encoding='utf-8')
result_filled.to_csv(p2, index=False, encoding='utf-8-sig')

print(open(p1, 'rb').read(6))   # 正常 UTF-8 中文的前几字节，如 b'\xe7\x94\xa8'
print(open(p2, 'rb').read(6))   # 开头多了 EF BB BF —— 这就是 BOM
```

**6. 类型修复**

```python
# 直接转 int 会失败，因为字符串"无订单"没法变成数字
try:
    result_filled['order_id'].astype(int)
except ValueError as e:
    print('失败：', e)

# 方案：先统一成字符串，语义最清楚（订单号本来也不是用来做算术的）
fixed = result_filled.copy()
fixed['order_id'] = fixed['order_id'].astype(str).str.replace(r'\.0$', '', regex=True)
print(fixed['order_id'].head())

# 如果想保留"可空的整数"语义，就先在合并阶段处理：
#   final_df['order_id'].astype('Int64')   # 能存 <NA>，且是整数
```
